# RGN — Colab

Rasyonel Çekim Ağı deneyleri. Eğitimler **arka planda** çalışır (`colab/runner.py`); hücreler hemen döner, durumu `R.nabiz()` ile izlersiniz. Ayrıntı: `colab/README.md`.

## 1. Hazırlık

In [ ]:
# Kod git'ten (önce commit + push), veri Drive'dan.  Deneme kolu için BRANCH'i değiştirin (örn. 'deneme/rgn-asama0').
REPO, BRANCH = 'https://github.com/sekerahmet/sekerai.git', 'main'
import torch; assert torch.cuda.is_available(), 'GPU yok: Çalışma zamanı > Çalışma zamanı türünü değiştir'
from google.colab import drive
drive.mount('/content/drive')
!git clone -q --depth 1 --branch {BRANCH} {REPO} /content/_repo && python /content/_repo/rgn/colab/setup_colab.py --git {REPO} --branch {BRANCH} --subdir rgn
import sys; sys.path.insert(0, '/content/RGN/colab'); import runner as R
%cd /content/RGN
!git -C /content/_repo log --oneline -1

## 2. Testler (CUDA testleri dahil) ve veri durumu

In [ ]:
!python tests/run_tests.py
!python scripts/prepare_data.py --list

## 3. Duman testi (~300 adım, arka planda)

In [ ]:
R.kos('configs/ss/ss_smoke.json')

In [ ]:
R.nabiz('ss_smoke', satir=20)   # tekrar çalıştırarak izleyin

## 4. SS karşılaştırma deneyleri (Aşama 0, sırayla, arka planda)

Her koşu 1 epok ve sonunda otomatik hız profili (`speed.json`). Oturum düşerse aynı hücre `--resume` ile kaldığı yerden sürer.

In [ ]:
R.kuyruk([
    'configs/ss/ss_d512_rgn.json',              # ana model
    'configs/ss/ss_d512_transformer.json',      # karşılaştırma tabanı
    'configs/ss/ss_d512_rgn_swiglu.json',       # yalnız MLP değişir
    'configs/ss/ss_d512_softmax_rational.json', # yalnız dikkat değişir
    'configs/ss/ss_d512_rgn_rope.json',         # konum: RoPE
    'configs/ss/ss_d512_rgn_alibi.json',        # konum: çarpımsal β
    'configs/ss/ss_d512_rgn_p2.json',           # çekim üssü: ||q̂-k̂||⁴
    'configs/ss/ss_d512_rgn_bf16.json',         # hassasiyet: çekim dikkati bf16
], name='ss_asama0')

In [ ]:
R.nabiz()

In [ ]:
# R.kapat('ss_asama0')   # gerekirse: checkpoint kaydedip durdurur

## 5. Hız profili (dikkat çekirdekleri, arka planda)

In [ ]:
R.arkaplan('hiz', 'python scripts/bench_speed.py attention --seq 512 1024 2048 4096 --batch 16')

## 6. Sonuç tablosu

In [ ]:
!python scripts/summarize_runs.py